In [3]:
import os
import torch

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model

In [4]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [5]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [6]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 20
eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1= model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Input Norm: torch.Size([13])|torch.Size([13])
Target Norm: torch.Size([4])|torch.Size([4])
3181824
5632
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 21:

Finished
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 21:

Finished


## Model Analysis -------------------------

In [7]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


|█████▎    | 53.7% (00:40) Evaluating model on validation data... (612/613) [3226/6003]:                  

RuntimeError: Error(s) in loading state_dict for StockGPT:
	Unexpected key(s) in state_dict: "transformer_blocks.4.ln1.scale", "transformer_blocks.4.ln1.shift", "transformer_blocks.4.mha.W_q.weight", "transformer_blocks.4.mha.W_k.weight", "transformer_blocks.4.mha.W_v.weight", "transformer_blocks.4.mha.out_proj.weight", "transformer_blocks.4.mha.out_proj.bias", "transformer_blocks.4.ln2.scale", "transformer_blocks.4.ln2.shift", "transformer_blocks.4.ff.layers.0.weight", "transformer_blocks.4.ff.layers.0.bias", "transformer_blocks.4.ff.layers.2.weight", "transformer_blocks.4.ff.layers.2.bias", "transformer_blocks.5.ln1.scale", "transformer_blocks.5.ln1.shift", "transformer_blocks.5.mha.W_q.weight", "transformer_blocks.5.mha.W_k.weight", "transformer_blocks.5.mha.W_v.weight", "transformer_blocks.5.mha.out_proj.weight", "transformer_blocks.5.mha.out_proj.bias", "transformer_blocks.5.ln2.scale", "transformer_blocks.5.ln2.shift", "transformer_blocks.5.ff.layers.0.weight", "transformer_block

|█████▎    | 53.7% (00:59) Evaluating model on validation data... (612/613) [3226/6003]: 

In [ ]:
for key, features in [("NLL", StockGPT_cfg["target_features"]), 
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]), 
                      ("MAE", StockGPT_cfg["target_features"]), 
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses 
                                + linear_losses + linear_test_losses 
                                + naive_losses + naive_test_losses, key),
                ["StockGPT", "LinearModel", "NaiveModel"],
                [f"{format_num(stockGPT_params)}", f"{format_num(linearModel_params)}", f"0"],
                features, key)